In [ ]:
# SINGLE CELL: Full PSI/PSR annotation pipeline with:
# (1) streamer RAG profiles, (2) local batch context, (3) resumable JSON updates,
# (4) Adaptive Prefix Batching (token-feasible K_token + EWMA health + AIMD + cooldown + retries),
# (5) XLSX export + total time + total cost (input/cached/output pricing).
#
# CHANGES APPLIED:
# (1) Working copy: messages.json -> 8_51_Enhanced_Contexts.json (all writes go there) + XLSX named 8_51_Enhanced_Contexts.xlsx
# (2) Per-batch cumulative tokens/cost/time printed + resumable stats file: 8_51_Enhanced_Contexts_stats_run.json
# (3) Replace latency-based T with output-pressure / truncation-based T:
#     T=1 if (output_tokens / M_out) >= PI_OUTPUT or looks_truncated_json(output_text) is True
# (4) Streamer profile logging clearly indicates loaded vs web_search creation

import os, json, time, re, sys, math, shutil
from datetime import datetime
from typing import Dict, Any, List, Tuple

# -----------------------------
# 0) Install/import deps
# -----------------------------
def _pip_install(pkgs: List[str]):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

try:
    import pandas as pd
except Exception:
    _pip_install(["pandas", "openpyxl"])
    import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    _pip_install(["python-dotenv"])
    from dotenv import load_dotenv

try:
    import tiktoken
except Exception:
    _pip_install(["tiktoken"])
    import tiktoken

try:
    from openai import OpenAI
except Exception:
    _pip_install(["openai"])
    from openai import OpenAI

# -----------------------------
# 1) Config
# -----------------------------
ROOT_DIR = os.getcwd()

RUN_BASENAME          = "PPCCOT"
SOURCE_MESSAGES_PATH  = os.path.join(ROOT_DIR, "messages.json")
WORK_MESSAGES_PATH    = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.json")
STATS_PATH            = os.path.join(ROOT_DIR, f"{RUN_BASENAME}_stats_run.json")

PROFILES_DIR  = os.path.join(ROOT_DIR, "streamer_profiles")
XLSX_OUT      = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.xlsx")
os.makedirs(PROFILES_DIR, exist_ok=True)

ANNOTATION_MODEL = "gpt-5.1"   
PROFILE_MODEL    = "gpt-5.1"   

# Token / context knobs
C_CONTEXT_TOKENS    = 400_000
ALPHA               = 0.1
OUTPUT_TOKEN_BUDGET = 4_000     # M_out
OVERHEAD_BUFFER     = 1_500     # B_safe
MAX_K               = 60

# Output-pressure threshold π
PI_OUTPUT = 0.90

SAVE_EVERY_BATCH  = True
PRINT_EVERY_BATCH = True

BINARY_FIELDS = [
    "DirectAddress", "Attachment", "SelfDisclose", "ReplySeeking",
    "CommRitual", "Monetary", "Backseat", "Emotes"
]
OTHER_FIELDS = ["Dimension"]

# Adaptive Prefix Batching: controller defaults (per your spec)
WARMUP_K0          = 20
WARMUP_B0          = 10   # number of successful warmup batches
RHO0               = 0.30
BETA_EWMA          = 0.90
DELTA_AIMD         = 1
GAMMA_AIMD         = 0.70
N_COOLDOWN         = 3
K_MIN              = 1
TAU_EWMA           = 0.10  # target health threshold (tunable)

# Health loss weights
W_F, W_T, W_V = 10.0, 3.0, 0.0

# Retry semantics
R_MAX = 2  # up to 2 retries => max 3 attempts total per pointer

# Pricing (per 1M tokens)
PRICE_INPUT_PER_1M   = 1.25
PRICE_CACHED_PER_1M  = 0.125
PRICE_OUTPUT_PER_1M  = 10.00

# Optional compact suffix for retries (helps truncation / verbosity)
COMPACT_SUFFIX = "\n\nCRITICAL: Output MUST be valid JSON only (no code fences)."

# -----------------------------
# 2) Env + client
# -----------------------------
load_dotenv(os.path.join(ROOT_DIR, ".env"))
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Ensure .env exists with OPENAI_API_KEY=...")

client = OpenAI()

# -----------------------------
# 3) Helpers (I/O, tokens, printing, usage+cost)
# -----------------------------
def now_ts():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def safe_write_json(path: str, obj: Any):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def ensure_working_copy():
    if os.path.exists(WORK_MESSAGES_PATH):
        return
    if not os.path.exists(SOURCE_MESSAGES_PATH):
        raise FileNotFoundError(f"Source messages.json not found at: {SOURCE_MESSAGES_PATH}")
    shutil.copyfile(SOURCE_MESSAGES_PATH, WORK_MESSAGES_PATH)
    print(f"[{now_ts()}] WORKING COPY: created {WORK_MESSAGES_PATH} from {SOURCE_MESSAGES_PATH}")

def load_messages() -> List[Dict[str, Any]]:
    if not os.path.exists(WORK_MESSAGES_PATH):
        raise FileNotFoundError(f"Working messages json not found at: {WORK_MESSAGES_PATH}")
    with open(WORK_MESSAGES_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Working messages file must be a JSON array of message objects.")
    return data

def is_annotated(msg: Dict[str, Any]) -> bool:
    return str(msg.get("Annotated", "")).strip().lower() == "true"

def mark_annotated(msg: Dict[str, Any]):
    msg["Annotated"] = "true"

def ensure_fields(msg: Dict[str, Any]):
    for k in BINARY_FIELDS + OTHER_FIELDS:
        if k not in msg:
            msg[k] = ""
    for k in ["raw_msg", "msgTag", "streamer", "Annotated"]:
        if k not in msg:
            msg[k] = ""
    if msg["Annotated"] == "":
        msg["Annotated"] = "False"

def get_encoding():
    try:
        return tiktoken.encoding_for_model("gpt-4.1")
    except Exception:
        return tiktoken.get_encoding("o200k_base")

ENC = get_encoding()

def tok_len(s: str) -> int:
    return len(ENC.encode(s or ""))

def usage_breakdown(resp) -> Dict[str, Any]:
    u = getattr(resp, "usage", None)
    if not u:
        return {}
    try:
        return u if isinstance(u, dict) else u.model_dump()
    except Exception:
        try:
            return dict(u)
        except Exception:
            return {"raw_usage": str(u)}

def add_usage_and_cost(stats: Dict[str, Any], resp):
    ud = usage_breakdown(resp)
    if not ud:
        return

    in_tok  = int(ud.get("input_tokens") or 0)
    out_tok = int(ud.get("output_tokens") or 0)

    cached_tok = 0
    itd = ud.get("input_tokens_details") or {}
    if isinstance(itd, dict):
        cached_tok = int(itd.get("cached_tokens") or 0)

    uncached_in = max(0, in_tok - cached_tok)

    stats["total_input_tokens"]  += in_tok
    stats["total_output_tokens"] += out_tok
    stats["total_cached_tokens"] += cached_tok
    stats["total_requests"]      += 1

    cost = (uncached_in / 1_000_000) * PRICE_INPUT_PER_1M \
         + (cached_tok   / 1_000_000) * PRICE_CACHED_PER_1M \
         + (out_tok      / 1_000_000) * PRICE_OUTPUT_PER_1M
    stats["total_cost_usd"] += cost

def enforce_binary(v: Any) -> str:
    return "Y" if str(v).strip().upper() == "Y" else ""

def enforce_dimension(v: Any, fallback: str = "P") -> str:
    s = str(v).strip().upper()
    if s in ("P", "POS", "POSITIVE"):
        return "P"
    if s in ("N", "NEG", "NEGATIVE"):
        return "N"
    return fallback

def call_with_retries(kwargs, tries=5, base_sleep=1.5):
    last_err = None
    for attempt in range(1, tries + 1):
        try:
            return client.responses.create(**kwargs)
        except Exception as e:
            last_err = e
            sleep = base_sleep * (2 ** (attempt - 1))
            print(f"[{now_ts()}]   ERROR (attempt {attempt}/{tries}): {e}. Sleeping {sleep:.1f}s then retry...")
            time.sleep(sleep)
    raise last_err

def load_or_init_stats() -> Dict[str, Any]:
    if os.path.exists(STATS_PATH):
        try:
            with open(STATS_PATH, "r", encoding="utf-8") as f:
                s = json.load(f)
            if isinstance(s, dict):
                # harden expected keys
                s.setdefault("total_requests", 0)
                s.setdefault("total_input_tokens", 0)
                s.setdefault("total_output_tokens", 0)
                s.setdefault("total_cached_tokens", 0)
                s.setdefault("total_cost_usd", 0.0)
                s.setdefault("total_elapsed_seconds", 0.0)
                s.setdefault("batches_completed", 0)
                s.setdefault("batches", [])
                return s
        except Exception:
            pass
    return {
        "total_requests": 0,
        "total_input_tokens": 0,
        "total_output_tokens": 0,
        "total_cached_tokens": 0,
        "total_cost_usd": 0.0,
        "total_elapsed_seconds": 0.0,   # cumulative across resumes
        "batches_completed": 0,
        "batches": [],
    }

def current_elapsed(stats: Dict[str, Any], session_start_wall: float) -> float:
    return float(stats.get("total_elapsed_seconds", 0.0)) + (time.time() - session_start_wall)

def save_stats(stats: Dict[str, Any], session_start_wall: float):
    # persist cumulative elapsed up to now
    stats["total_elapsed_seconds"] = float(stats.get("total_elapsed_seconds", 0.0)) + (time.time() - session_start_wall)
    stats["last_saved_ts"] = now_ts()
    safe_write_json(STATS_PATH, stats)

def print_cum_stats_line(stats: Dict[str, Any], session_start_wall: float, prefix: str = "CUM"):
    in_tok  = int(stats.get("total_input_tokens", 0))
    out_tok = int(stats.get("total_output_tokens", 0))
    c_tok   = int(stats.get("total_cached_tokens", 0))
    unc_in  = max(0, in_tok - c_tok)
    cost_usd = float(stats.get("total_cost_usd", 0.0))
    el = current_elapsed(stats, session_start_wall)
    req = int(stats.get("total_requests", 0))
    print(f"[{now_ts()}]   {prefix}: req={req} | input={in_tok} (uncached≈{unc_in}, cached={c_tok}) | output={out_tok} | cost=${cost_usd:.4f} | elapsed_total={el:.2f}s")

# -----------------------------
# 4) JSON parsing / repair + truncation signal
# -----------------------------
def try_repair_json(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"^```\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    t = t.strip()

    m_obj = re.search(r"\{.*\}", t, flags=re.DOTALL)
    m_arr = re.search(r"\[.*\]", t, flags=re.DOTALL)
    if m_arr and (not m_obj or len(m_arr.group(0)) >= len(m_obj.group(0))):
        t = m_arr.group(0)
    elif m_obj:
        t = m_obj.group(0)

    t = t.replace("\t", " ")
    t = re.sub(r",\s*([}\]])", r"\1", t)
    t = re.sub(r"(?<!\\)\n", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t

def get_output_text(resp) -> str:
    if hasattr(resp, "output_text") and resp.output_text:
        return resp.output_text
    out = getattr(resp, "output", None) or []
    for item in out:
        if isinstance(item, dict) and item.get("type") == "message":
            for c in (item.get("content") or []):
                if isinstance(c, dict) and c.get("type") in ("output_text", "text"):
                    return c.get("text", "") or ""
    return ""

def looks_truncated_json(text: str) -> bool:
    """
    Heuristic truncation detector:
    - last non-space char not in } or ]
    - bracket/brace imbalance (more opens than closes)
    - obvious cutoff tokens near end
    """
    t = (text or "").strip()
    if not t:
        return False
    # strip code fences if present
    t2 = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE).strip()
    t2 = re.sub(r"^```\s*", "", t2).strip()
    t2 = re.sub(r"\s*```$", "", t2).strip()
    if not t2:
        return False

    last = t2[-1]
    if last not in ("]", "}"):
        return True

    opens = t2.count("{") + t2.count("[")
    closes = t2.count("}") + t2.count("]")
    if opens > closes:
        return True

    tail = t2[-60:].lower()
    if any(x in tail for x in ["\"dimension\":", "\"emotes\":", "\"directaddress\":"]) and tail.endswith((",", ":", "\"", "\\")):
        return True

    return False

def parse_json_result(resp) -> Dict[str, Any]:
    text = get_output_text(resp)
    if not text:
        raise RuntimeError("Empty model output; cannot parse JSON.")
    try:
        t = try_repair_json(text)
        result = json.loads(t)
    except Exception as e1:
        raise RuntimeError(f"JSON parse failed after repair: {e1}\n---RAW_START---\n{text[:2000]}\n---RAW_END---")
    if isinstance(result, list):
        return {"items": result}
    if isinstance(result, dict) and "items" in result:
        return result
    raise RuntimeError("Parsed JSON is neither a list nor an object with 'items'.")

# -----------------------------
# 5) Streamer profile (RAG via web_search) -> saved once per streamer
# -----------------------------
def safe_filename(s: str) -> str:
    s = (s or "_unknown").strip()
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)[:120]

def ensure_profile(streamer: str, stats: Dict[str, Any], session_start_wall: float) -> Dict[str, Any]:
    fn = os.path.join(PROFILES_DIR, f"{safe_filename(streamer)}.json")
    if os.path.exists(fn):
        with open(fn, "r", encoding="utf-8") as f:
            prof = json.load(f)
        print(f"[{now_ts()}] PROFILE: loaded {fn}")
        return prof

    print(f"[{now_ts()}] PROFILE: web_search start for streamer='{streamer}'")

    # (kept as reference; not enforced by SDK here)
    profile_schema = {
        "name": "streamer_profile",
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "streamer": {"type": "string"},
                "Name": {"type": "string"},
                "Nicknames": {"type": "string"},
                "Location": {"type": "string"},
                "Affiliation": {"type": "string"},
                "Friends": {"type": "string"},
                "Interests": {"type": "string"},
                "ContentStyle": {"type": "string"},
                "Controversies": {"type": "string"},
                "Summary500": {"type": "string"}
            },
            "required": ["streamer", "Summary500"]
        }
    }

    prompt = f"""
You are building a compact streamer profile for downstream Twitch-chat annotation.
Search the web as needed. Return JSON ONLY matching the schema.

Target streamer handle: {streamer}

Fill (short fragments; if uncertain -> ""):
Name, Nicknames, Location, Affiliation, Friends, Interests, ContentStyle, Controversies

Summary500: a single compact summary <= 500 characters.
Be conservative: do not guess.
"""

    t0 = time.time()
    resp = client.responses.create(
        model=PROFILE_MODEL,
        input=[{"role": "user", "content": prompt + "\n\nReturn ONLY valid JSON. No markdown, no code fences."}],
        tools=[{"type": "web_search"}],
        include=["web_search_call.action.sources"],
        reasoning={"effort": "low"},
        max_output_tokens=800
    )
    dt = time.time() - t0

    add_usage_and_cost(stats, resp)

    prof = None
    try:
        prof = json.loads(get_output_text(resp))
    except Exception:
        prof = None

    if prof is None or not isinstance(prof, dict):
        raise RuntimeError("Failed to parse streamer profile JSON.")

    if len(prof.get("Summary500", "")) > 500:
        prof["Summary500"] = prof["Summary500"][:500]
    prof["streamer"] = streamer

    safe_write_json(fn, prof)

    ud = usage_breakdown(resp)
    cached = None
    try:
        cached = (ud.get("input_tokens_details") or {}).get("cached_tokens", None)
    except Exception:
        cached = None

    print(f"[{now_ts()}] PROFILE: saved {fn} (latency={dt:.2f}s, cached_tokens={cached})")
    print_cum_stats_line(stats, session_start_wall, prefix="CUM_PROFILE")
    return prof

# -----------------------------
# 6) Runs + local batch context
# -----------------------------
def contiguous_runs_unannotated(messages: List[Dict[str, Any]]) -> List[Tuple[str, List[int]]]:
    runs = []
    i, n = 0, len(messages)
    while i < n:
        if is_annotated(messages[i]):
            i += 1
            continue
        streamer = (messages[i].get("streamer") or "").strip() or "_unknown"
        j = i
        idxs = []
        while j < n and (((messages[j].get("streamer") or "").strip() or "_unknown") == streamer):
            if not is_annotated(messages[j]):
                idxs.append(j)
            j += 1
        if idxs:
            runs.append((streamer, idxs))
        i = j
    return runs

def build_local_batch_context(packed: List[Dict[str, Any]], streamer: str) -> str:
    streamer_l = (streamer or "").lower().strip()
    texts = [(p.get("raw_msg","") or "") for p in packed]
    tags  = [(p.get("msgTag","") or "") for p in packed]

    tag_counts = {}
    for t in tags:
        t2 = t.strip().upper()
        tag_counts[t2] = tag_counts.get(t2, 0) + 1

    mentions = []
    for txt in texts:
        mentions += [m.lower() for m in re.findall(r"@\w{2,}", txt)]

    chat_addr_markers = ["chat", "you guys", "yall", "y'all", "everyone", "guys"]
    reply_markers = ["notice me", "read this", "answer", "pls", "please", "did you see", "remember me", "why you not", "reply"]
    argument_markers = ["youre wrong", "you're wrong", "idiot", "shut up", "stupid", "clown", "L take", "trash"]
    backseat_markers = ["go ", "use ", "buy ", "skip", "watch out", "turn on", "turn off", "in the menu", "settings", "press", "click"]
    emote_pattern = re.compile(r"(<3|:\)|;\)|:\(|:D|xd\b|kekw\b|kappa\b|pog\b|pogchamp\b|lul\b|wutface\b|monkas\b)", re.IGNORECASE)

    def has_any(txt, arr):
        tl = txt.lower()
        return any(a in tl for a in arr)

    streamer_name_ref = 0
    streamer_at_ref = 0
    second_person = 0
    chat_address = 0
    reply_bids = 0
    argument_like = 0
    backseat_like = 0
    emote_like = 0
    question_like = 0

    for txt in texts:
        tl = txt.lower()
        if streamer_l and re.search(rf"\b{re.escape(streamer_l)}\b", tl):
            streamer_name_ref += 1
        if streamer_l and f"@{streamer_l}" in tl:
            streamer_at_ref += 1
        if re.search(r"\b(you|your|u)\b", tl):
            second_person += 1
        if has_any(txt, chat_addr_markers):
            chat_address += 1
        if has_any(txt, reply_markers):
            reply_bids += 1
        if has_any(txt, argument_markers):
            argument_like += 1
        if has_any(txt, backseat_markers):
            backseat_like += 1
        if emote_pattern.search(txt) is not None:
            emote_like += 1
        if "?" in txt:
            question_like += 1

    n = max(1, len(texts))
    blob = " ".join(texts).lower()
    blob = re.sub(r"http\S+", " ", blob)
    blob = re.sub(r"[^a-z0-9_@<3\s]", " ", blob)
    toks = [t for t in blob.split() if len(t) >= 4 and not t.startswith("@")]
    stop = set(["this","that","with","have","just","like","they","them","then","what","when","your","youre","gonna","dont","does","from","about","really","would","there","here","also","chat"])
    toks = [t for t in toks if t not in stop]
    freq = {}
    for t in toks[:1200]:
        freq[t] = freq.get(t, 0) + 1
    top = sorted(freq.items(), key=lambda x: (-x[1], x[0]))[:10]
    top_kw = [k for k,_ in top]

    ctx = {
        "streamer": streamer,
        "batch_size": len(packed),
        "msgTag_mix": tag_counts,
        "notable_mentions": sorted(set(mentions))[:12],
        "salient_terms": top_kw,
        "rate_streamer_name_ref": round(streamer_name_ref / n, 3),
        "rate_streamer_at_ref": round(streamer_at_ref / n, 3),
        "rate_second_person": round(second_person / n, 3),
        "rate_chat_address": round(chat_address / n, 3),
        "rate_questions": round(question_like / n, 3),
        "rate_reply_bids": round(reply_bids / n, 3),
        "rate_argument_like": round(argument_like / n, 3),
        "rate_backseat_like": round(backseat_like / n, 3),
        "rate_emotes": round(emote_like / n, 3),
        "note": "Scan INPUT_MESSAGES as a time-contiguous window; use these signals only to disambiguate addressee/topic; if unclear, prefer blanks for high precision."
    }
    return json.dumps(ctx, ensure_ascii=False)

# -----------------------------
# 7) Developer prompt + schema placeholders
# -----------------------------
DEVELOPER_PROMPT = r"""
You are an annotation model. Your task is to label each Twitch chat message with a set of binary/ternary fields using the provided schema.
Follow definitions and decision rules exactly. Return ONLY a JSON array (no wrapper object, no text).

OUTPUT FORMAT (STRICT)
Return ONLY a JSON array with EXACTLY the same number of elements as INPUT_MESSAGES, in the same order.
Each element must be an object with keys:
id, DirectAddress, Attachment, SelfDisclose, ReplySeeking, CommRitual, Monetary, Backseat, Emotes, Dimension (strings).

Each label field must be one of:
"Y" = present
""  = not present / leave blank

For Dimension, use exactly one of:
"P" if positive/affiliative stance toward streamer/community
"N" if negative/hostile stance toward streamer/community
No neutral/mixed category. If mixed, label based on the dominant intent (“punchline”).

WHAT YOU ARE ANNOTATING
Annotate message-level expressed cues in livestream chat from Twitch. Multi-label is allowed.
Do not infer hidden intent beyond the text and immediate conversational form.

TWO-LAYER CONTEXT YOU MUST USE
You will receive:
1) STREAMER_GENERAL_CONTEXT: background about the streamer (name, common nicknames, friends, recurring memes, controversies, locations, typical content, etc.).
   Use it only to resolve reference/ambiguity (e.g., who “you” refers to; whether a named person is the streamer vs. a friend).
2) LOCAL_BATCH_CONTEXT: what is being discussed in this particular batch (topic/game/event, current narrative, who is speaking to whom).
   Use it to disambiguate pronouns and whether a question is aimed at the streamer vs. the chat and topic.

If the general + local contexts still do not make the addressee clear, prefer leaving categories blank (high precision).

Before labeling individual messages, quickly scan the entire INPUT_MESSAGES batch to infer LOCAL_BATCH_CONTEXT signals (topic, who “you” refers to, whether chatters are arguing, and whether questions target streamer vs chat), then annotate each message.

========================================================
CCOT: CONTRASTIVE CHAIN-OF-THOUGHT (INTERNAL ONLY; DO NOT OUTPUT)
You MUST follow this internal reasoning protocol, but you MUST NOT output any rationale, notes, or intermediate reasoning—ONLY the final JSON array.

For each message i:
1) Establish two competing interpretations (contrast set):
   - H_streamer: message targets the streamer.
   - H_chat/other: message targets chat or another person.
   Use STREAMER_GENERAL_CONTEXT + LOCAL_BATCH_CONTEXT to choose the more plausible hypothesis.
   If neither is clearly favored, treat it as ambiguous and bias toward blanks for addressee-dependent labels.

2) For EACH label L in {DirectAddress, Attachment, SelfDisclose, ReplySeeking, CommRitual, Monetary, Backseat, Emotes}:
   Internally generate a minimal contrastive check:
   - Evidence FOR "Y" (what exact token/phrase/form triggers it)
   - Evidence AGAINST "Y" (what exact token/phrase/form blocks it)
   Then decide "Y" only if FOR clearly dominates AGAINST under the rules (high precision).
   If uncertain, output "".

3) For Dimension:
   Internally contrast P vs N by locating the “punchline” or dominant intent:
   - P if dominant stance is support/affection/encouragement (including clearly supportive teasing).
   - N if dominant stance is hostility/insult/shaming/threat/aggressive blame.
   If mixed, choose the dominant thrust.

4) Final consistency check (internal):
   - Do not label DirectAddress unless addressee is strongly the streamer (name/@mention or clear local context).
   - Monetary override: msgTag USERNOTICE => Monetary="Y" ALWAYS.
   - Emotes="Y" if any Twitch-style emote tokens or unicode emojis appear (including <3).
   Output only the labels, never the reasoning.
========================================================

CATEGORY DECISION RULES
A) Dimension (valence)
Set Dimension independently from other categories.
P (Positive): praise/support, gratitude, affection, encouragement, admiration; playful teasing that is clearly supportive.
N (Negative): insults, hostility, shaming, threats, aggressive blame; hate-watch enjoyment of failure.
Mixed: choose dominant intent.

B) PSI/PSR-leaning categories (multi-label)
1) Direct Address / second-person to streamer -> DirectAddress
Mark "Y" only when the message targets the streamer as an addressee:
- explicit @streamer
- streamer name used as addressee (“Emiru, …”)
- second-person pronouns clearly referring to streamer not another person (“you/your/u”) with directed statement/command
Avoid false positives:
- talking about streamer: “emiru is cracked”
- addressing chat: “you guys…”
- ambiguous “you” -> blank unless strong cue (name/@mention or clear local context).

Important Point! Keep an eye on morphs, euphemisms, and varying forms of chatting common for Gen-Z, like "ya", "yo", "u" etc. to better understand addressees.

2) Attachment / affection / relational warmth -> Attachment
Mark "Y" for warmth/care/pride/missing/love toward streamer or relational closeness markers.
Attachment-strengtheners (no extra labels; just annotate Attachment when applicable):
- longitudinal persistence beyond exposure: off-stream anticipation, absence felt, “all week,” “can’t wait,” “glad you’re back”
- investment/loyalty talk: “watch every stream,” “since 2018,” “never miss a VOD,” schedule organized around streams
- social realism / real-life tie framing: “you’re like family,” “real friend,” “like a brother/sister”
- anthropomorphism/personification: “you care about us,” “she understands me,” “he worries about chat”
Avoid false positives:
- “love this game” (not streamer-directed)
- generic hype alone (“Pog”, “Lets go”) unless clearly relational.

3) Identification / self-disclosure / homophily -> SelfDisclose
Mark "Y" when viewer shares personal info or similarity claims aimed at relational connection:
- “I’m also…”, “same here…”, “as a fellow…”, "I also do that..."
- personal disclosure positioned as sharing-with-streamer (“I had a rough day, thanks for being here”)
- “I have an exam tomorrow, wish me luck” can count if framed as a bid for connection (even without explicit “you” if local context makes it streamer-directed).
Strengthener (no new label):
- knowledge/narrative recall is not SelfDisclose by itself; use it to support DirectAddress/Attachment/ReplySeeking if it functions that way.

4) Reciprocity bids / reply-seeking -> ReplySeeking
Mark "Y" when seeking acknowledgment/response/recognition:
- “notice me,” “read this,” “answer please,” “remember me,” “did you see my last message,” “thanks for replying last week,” "why you not seeing me"
- Or asking them to do something ordinary like drinking water or blink for safety of eyes.
Avoid false positives:
- generic info questions not aimed at streamer (“what game is this?”) unless streamer-directed by name/@mention or clear local context.

5) Community ritual talk & co-creation -> CommRitual
Mark "Y" for shared rituals/inside jokes/coordinated norms tied to community:
- “spam LUL,” “copy pasta time,” “chat do the thing,” “only real ones remember…,” “as always…,” "Let's go..."
Avoid false positives:
- emotes alone ≠ ritual talk without ritual framing.

6) Monetary support actions -> Monetary
Mark "Y" for subs/donos/gifts/streaks or accompanying text:
- “subbed X months,” “gifted subs,” “dono,” “renewed,” “superchat”
Hard rule: If the message tag is "USERNOTICE" instead of "PRIVATEMESSAGE" ALWAYS mark Monetary="Y".
Avoid false positives:
- unrelated money talk (“costs $60”).

ADDED CATEGORIES (behavioral / surface form)
7) Backseat guidance -> Backseat
Mark "Y" for gameplay/stream/content direction/coaching/strategy/idea/suggestion:
- “go left,” “use ult,” “buy armor,” “skip cutscene,” “watch out…,” "You have to change it in the menu"
Or general questions asking regarding the topic:
- "Hassan are they capitalists?", "you should do a comparison on their technical data"
Backseat can co-occur with DirectAddress, but backseating is not parasocial by itself.

8) Emotes/emoji presence -> Emotes
Mark "Y" if any Twitch-style emote tokens or unicode emojis appear, including <3, and tokens like Kappa, pogchamp, KEKW, etc.

CONTRASTIVE DEMONSTRATIONS (BEHAVIORAL TARGETS)
Follow the demos for common pitfalls:

Demo A — DirectAddress vs about-talk; “you” ambiguity
General: emiru is the streamer; “chat” = audience. Local: streamer just returned; greetings happening.
Inputs:
(1) "Everyone missed you beautiful Emiru <3 <3 WutFace"
(2) "emiru is cracked today"
(3) "you guys missed her too right? lol"
Correct:
(1) DirectAddress=Y (name + “you”), Attachment=Y (“missed you”, warmth), Emotes=Y, Dimension=P
(2) About-talk only: DirectAddress=""; Dimension=P (praise)
(3) Targets chat (“you guys”): DirectAddress=""; Dimension=P (light/affiliative)
Avoid:
- Don’t set DirectAddress just because streamer name appears (2)
- Don’t set DirectAddress from “you” when it clearly addresses chat (3)

Demo B — Loyalty/investment as Attachment; recall ≠ SelfDisclose; mixed valence punchline rule
General: hasanabi political streams; “since YEAR” indicates tenure.
Inputs:
(10) "Been watching since 2019. Never miss a VOD."
(11) "Remember that 2019 meltdown? downhill ever since lol"
(12) "I love you but you’re being an idiot tonight"
Correct:
(10) Attachment=Y (investment/loyalty), SelfDisclose=Y, Dimension=P
(11) Dimension=N (criticism); SelfDisclose="" (recall is not personal disclosure)
(12) Attachment=Y (“love you”), Dimension=N (dominant thrust is insult/punchline)
Avoid:
- Don’t force SelfDisclose from narrative recall (11)
- Don’t label Dimension=P just because affection phrase exists (12)

Demo C — USERNOTICE monetary override + co-occurrence
Rule: msgTag="USERNOTICE" => Monetary=Y ALWAYS.
Input:
(20) msgTag=USERNOTICE, "Renewed my sub for 24 months — love you Emiru <3"
Correct:
Monetary=Y (forced), Attachment=Y, DirectAddress=Y (name used as addressee), Emotes=Y (<3), Dimension=P
Avoid:
- Never leave Monetary blank on USERNOTICE even if message is mostly affection

FINAL OUTPUT CONSTRAINTS
Return ONLY a JSON array, no text, no code fences.
Use the provided id values exactly as given in INPUT_MESSAGES (0..K-1). Do not renumber or reorder items.
Do NOT include any explanations, rationales, “because…”, confidence scores, or extra keys beyond the required keys.
"""

# -----------------------------
# 7) Output schema (kept as in your code; not used in SDK call)
# -----------------------------
annotation_schema = {
    "name": "psi_annotation_batch",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "id": {"type": "integer"},
                        "DirectAddress": {"type": "string"},
                        "Attachment": {"type": "string"},
                        "SelfDisclose": {"type": "string"},
                        "ReplySeeking": {"type": "string"},
                        "CommRitual": {"type": "string"},
                        "Monetary": {"type": "string"},
                        "Backseat": {"type": "string"},
                        "Emotes": {"type": "string"},
                        "Dimension": {"type": "string"}
                    },
                    "required": ["id","DirectAddress","Attachment","SelfDisclose","ReplySeeking","CommRitual","Monetary","Backseat","Emotes","Dimension"]
                }
            }
        },
        "required": ["items"]
    }
}

# -----------------------------
# 8) Adaptive Prefix Batching (Token feasibility + EWMA + AIMD)
# -----------------------------
def C_in_budget() -> int:
    # C_in = alpha*C - M_out - B_safe
    return int(ALPHA * C_CONTEXT_TOKENS) - int(OUTPUT_TOKEN_BUDGET) - int(OVERHEAD_BUFFER)

def _to_compact_str(v) -> str:
    if v is None:
        return ""
    if isinstance(v, str):
        return v.strip()
    if isinstance(v, (list, tuple)):
        # join list items cleanly
        parts = []
        for x in v:
            xs = _to_compact_str(x)
            if xs:
                parts.append(xs)
        return ", ".join(parts).strip()
    if isinstance(v, dict):
        # small dict -> json on one line
        try:
            return json.dumps(v, ensure_ascii=False, separators=(",", ":")).strip()
        except Exception:
            return str(v).strip()
    # numbers/bools/etc
    return str(v).strip()

def compact_streamer_context(prof: Dict[str, Any]) -> str:
    fields = ["streamer","Name","Nicknames","Location","Affiliation","Friends",
              "Interests","ContentStyle","Controversies","Summary500"]
    lines = []
    for k in fields:
        v = _to_compact_str(prof.get(k, ""))
        if v:
            lines.append(f"{k}: {v}")
    return "\n".join(lines)

def pack_items(messages: List[Dict[str, Any]], take: List[int]) -> Tuple[List[Dict[str, Any]], Dict[int, int]]:
    packed = []
    id_to_global = {}
    for j, gi in enumerate(take):
        packed.append({
            "id": j,
            "msgTag": messages[gi].get("msgTag", ""),
            "raw_msg": messages[gi].get("raw_msg", ""),
            #"streamer": messages[gi].get("streamer", "")
        })
        id_to_global[j] = gi
    return packed, id_to_global

def build_user_content(local_ctx: str, packed: List[Dict[str, Any]], compact: bool=False) -> str:
    base = (
        "LOCAL_BATCH_CONTEXT:\n<<<" + local_ctx + ">>>\n\n"
        "INPUT_MESSAGES (array of objects with id, raw_msg, msgTag, optional streamer):\n<<<"
        + json.dumps(packed, ensure_ascii=False)
        + ">>>"
        "\n\nReturn ONLY JSON."
    )
    if compact:
        base += COMPACT_SUFFIX
    return base

def H_static_tokens(streamer_general_context: str, dev_prompt: str, local_ctx_max: str) -> int:
    system_content = f"STREAMER_GENERAL_CONTEXT:\n{streamer_general_context}\n"
    user_prefix = (
        "LOCAL_BATCH_CONTEXT:\n<<<" + local_ctx_max + ">>>\n\n"
        "INPUT_MESSAGES (array of objects with id, raw_msg, msgTag, optional streamer):\n<<<"
    )
    user_suffix = ">>>\n\nReturn ONLY JSON."
    return tok_len(system_content) + tok_len(dev_prompt) + tok_len(user_prefix) + tok_len(user_suffix)

def K_token_cap(messages: List[Dict[str, Any]], idxs: List[int], streamer: str, streamer_general_context: str, dev_prompt: str) -> int:
    """
    K_token(i) = max K such that T_in(K) <= C_in.
    Conservative implementation: compute LOCAL context on max_window (<=MAX_K), use it in H.
    Then test feasible K by adding packed-list token estimates.
    """
    Cin = C_in_budget()
    if Cin <= 0:
        return 1

    max_window = min(len(idxs), MAX_K)
    if max_window <= 0:
        return 1

    packed_max, _ = pack_items(messages, idxs[:max_window])
    local_ctx_max = build_local_batch_context(packed_max, streamer)

    H = H_static_tokens(streamer_general_context, dev_prompt, local_ctx_max)
    if H >= Cin:
        return 1

    k_best = 1
    for k in range(1, max_window + 1):
        packed_k, _ = pack_items(messages, idxs[:k])
        Tin = H + tok_len(json.dumps(packed_k, ensure_ascii=False))
        if Tin <= Cin:
            k_best = k
        else:
            break
    return max(1, k_best)

def validate_item(it: Dict[str, Any]) -> int:
    if not isinstance(it, dict):
        return 0
    if "id" not in it:
        return 0
    for f in BINARY_FIELDS:
        v = it.get(f, "")
        if str(v).strip().upper() not in ("", "Y"):
            return 0
    dim = str(it.get("Dimension", "")).strip().upper()
    if dim not in ("P", "N"):
        return 0
    return 1

def health_loss(F: int, T: int, Y: float) -> float:
    denom = (W_F + W_T + W_V)
    if denom <= 0:
        return 0.0
    return float((W_F * F + W_T * T + W_V * (1.0 - Y) * (1 - F)) / denom)

def ewma_update(gbar_prev: float, g: float) -> float:
    return float(BETA_EWMA * gbar_prev + (1.0 - BETA_EWMA) * g)

def aimd_update(K: int, hard_event: bool, gbar: float, cooldown: int) -> Tuple[int, int]:
    """
    - immediate backoff on hard events (F=1 or T=1)
    - else cooldown hold
    - else if gbar <= tau, additive increase
    - else multiplicative decrease
    """
    if hard_event:
        K2 = max(K_MIN, int(math.floor(GAMMA_AIMD * K)))
        return K2, N_COOLDOWN

    if cooldown > 0:
        return K, cooldown - 1

    if gbar <= TAU_EWMA:
        return K + DELTA_AIMD, 0

    K2 = max(K_MIN, int(math.floor(GAMMA_AIMD * K)))
    return K2, N_COOLDOWN

# -----------------------------
# 9) Apply results robustly (ID mapping + positional fallback)
# -----------------------------
def apply_batch_results(messages: List[Dict[str, Any]], items: List[Dict[str, Any]], id_to_global: Dict[int, int]) -> int:
    def apply_to_global(gi: int, it: dict):
        for f in BINARY_FIELDS:
            messages[gi][f] = enforce_binary(it.get(f, ""))

        if str(messages[gi].get("msgTag", "")).strip().upper() == "USERNOTICE":
            messages[gi]["Monetary"] = "Y"

        raw = (messages[gi].get("raw_msg", "") or "").lower()
        fallback = "N" if re.search(r"\b(idiot|stupid|shut up|trash|hate|kill|die)\b", raw) else "P"
        messages[gi]["Dimension"] = enforce_dimension(it.get("Dimension", ""), fallback=fallback)
        mark_annotated(messages[gi])

    # Off-by-one fix (common)
    try:
        ids_int = [int(it.get("id")) for it in items if isinstance(it, dict) and "id" in it]
        if len(ids_int) == len(id_to_global) and set(ids_int) == set(range(1, len(id_to_global) + 1)):
            for it in items:
                it["id"] = int(it["id"]) - 1
    except Exception:
        pass

    updated = 0
    bad_id = 0

    # 1) ID mapping
    for it in items:
        if not isinstance(it, dict) or "id" not in it:
            bad_id += 1
            continue
        try:
            local_id = int(it["id"])
        except Exception:
            bad_id += 1
            continue
        gi = id_to_global.get(local_id)
        if gi is None:
            bad_id += 1
            continue
        apply_to_global(gi, it)
        updated += 1

    # 2) Positional fallback if ID mapping failed badly
    if updated < max(1, len(id_to_global) // 2):
        print(f"[{now_ts()}]   WARNING: id mapping weak (updated={updated}/{len(id_to_global)}). Using positional fallback.")
        updated = 0
        L = min(len(items), len(id_to_global))
        for j in range(L):
            it = items[j]
            if not isinstance(it, dict):
                continue
            gi = id_to_global.get(j)
            if gi is None:
                continue
            apply_to_global(gi, it)
            updated += 1

    return updated

# -----------------------------
# 10) Main execution
# -----------------------------
ensure_working_copy()
messages = load_messages()
for m in messages:
    ensure_fields(m)

total = len(messages)
remaining = sum(1 for m in messages if not is_annotated(m))
print(f"[{now_ts()}] Loaded {total} messages from WORK file. Remaining unannotated: {remaining}")
print(f"[{now_ts()}] WORK file:  {WORK_MESSAGES_PATH}")
print(f"[{now_ts()}] STATS file: {STATS_PATH}")
print(f"[{now_ts()}] XLSX out:    {XLSX_OUT}")

stats = load_or_init_stats()
session_start_wall = time.time()

# Controller state (global) — re-init on each run (messages + stats provide resumability)
controller = {
    "warmed": False,
    "warmup_successes": 0,
    "warmup_per_item_list_tokens": [],
    "K": WARMUP_K0,
    "cooldown": 0,
    "gbar": 0.0,
}

batch_counter = int(stats.get("batches_completed", 0))
newly_annotated = 0

print_cum_stats_line(stats, session_start_wall, prefix="CUM_START")

while True:
    remaining = sum(1 for m in messages if not is_annotated(m))
    if remaining == 0:
        break

    runs = contiguous_runs_unannotated(messages)
    if not runs:
        break

    streamer, idxs = runs[0]

    # RAG profile
    prof = ensure_profile(streamer, stats, session_start_wall)
    #streamer_general_context = json.dumps(prof, ensure_ascii=False)
    streamer_general_context = compact_streamer_context(prof)

    # Compute K_token(i) (conservative)
    Ktok = K_token_cap(messages, idxs, streamer, streamer_general_context, DEVELOPER_PROMPT)
    Ktok = max(1, min(Ktok, MAX_K))

    # Choose K for next attempt (warmup uses fixed K0; after warmup uses controller.K)
    if not controller["warmed"]:
        K_target = min(WARMUP_K0, Ktok)
    else:
        K_target = min(int(controller["K"]), Ktok)
    K_target = max(1, min(K_target, MAX_K))

    # For logging only
    take0 = idxs[:K_target]
    packed0, _ = pack_items(messages, take0)
    local_ctx0 = build_local_batch_context(packed0, streamer)

    if PRINT_EVERY_BATCH:
        print(f"\n[{now_ts()}] BATCH #{batch_counter+1} | streamer='{streamer}' | K_target={K_target} | K_token={Ktok} | remaining={remaining}")
        if not controller["warmed"]:
            print(f"[{now_ts()}]   warmup: {controller['warmup_successes']}/{WARMUP_B0} successes | current_K={controller['K']}")
        else:
            print(f"[{now_ts()}]   controller: K={controller['K']} cooldown={controller['cooldown']} gbar={controller['gbar']:.4f}")
        print_cum_stats_line(stats, session_start_wall, prefix="CUM_BEFORE_BATCH")

    # Attempt loop (retries on same pointer with smaller K)
    attempt = 0
    success = False

    last_F = 1
    last_T = 0
    last_Y = 0.0
    last_g = 1.0
    last_dt = 0.0
    last_items = None
    last_id_to_global = None
    last_packed_len = 0
    last_out_tok = 0
    last_out_text = ""

    while attempt < (R_MAX + 1) and not success:
        attempt += 1

        K_attempt = max(1, min(K_target, Ktok, MAX_K))
        take = idxs[:K_attempt]
        packed, id_to_global = pack_items(messages, take)
        local_ctx = build_local_batch_context(packed, streamer)

        compact = (attempt >= 2)  # add compact suffix on retries
        kwargs = dict(
            model=ANNOTATION_MODEL,
            input=[
                {"role": "system",    "content": f"STREAMER_GENERAL_CONTEXT:\n{streamer_general_context}\n"},
                {"role": "developer", "content": DEVELOPER_PROMPT},
                {"role": "user",      "content": build_user_content(local_ctx, packed, compact=compact)},
            ],
            tools=[],
            reasoning={"effort": "low"},
            max_output_tokens=OUTPUT_TOKEN_BUDGET
        )

        t0 = time.time()
        resp = call_with_retries(kwargs)
        dt = time.time() - t0
        last_dt = dt

        # usage + cost (for T and printing)
        add_usage_and_cost(stats, resp)
        ud = usage_breakdown(resp)
        out_tok = int((ud.get("output_tokens") or 0) if isinstance(ud, dict) else 0)
        last_out_tok = out_tok

        out_text = get_output_text(resp)
        last_out_text = out_text

        # Observe F (usability) and Y (validity, if usable)
        F = 0
        Y = 1.0
        items = None
        try:
            result = parse_json_result(resp)
            items = result.get("items", [])
            if not isinstance(items, list):
                raise RuntimeError("Parsed 'items' is not a list.")
            if len(items) > 0:
                Y = sum(validate_item(it) for it in items) / float(len(items))
            else:
                Y = 0.0
        except Exception as e:
            F = 1
            Y = 0.0
            if PRINT_EVERY_BATCH:
                print(f"[{now_ts()}]   attempt {attempt}: PARSE_FAIL (F=1) at K={K_attempt} | err={str(e)[:180]}")

        # NEW: T = output pressure OR truncation heuristic (no latency constraint)
        output_pressure = (out_tok / float(OUTPUT_TOKEN_BUDGET)) if OUTPUT_TOKEN_BUDGET > 0 else 0.0
        trunc = looks_truncated_json(out_text)
        T = 1 if (output_pressure >= PI_OUTPUT or trunc) else 0

        # Health loss g_t and EWMA update
        g = health_loss(F=F, T=T, Y=Y)
        controller["gbar"] = ewma_update(controller["gbar"], g)

        # AIMD update after this attempt
        hard_event = (F == 1) or (T == 1)
        K_next, cooldown_next = aimd_update(K=K_attempt, hard_event=hard_event, gbar=controller["gbar"], cooldown=int(controller["cooldown"]))
        K_next = max(1, min(K_next, Ktok, MAX_K))

        # Logging attempt summary
        if PRINT_EVERY_BATCH:
            print(f"[{now_ts()}]   attempt {attempt}: K={K_attempt} latency={dt:.2f}s "
                  f"out_tok={out_tok} P={output_pressure:.2f} trunc={int(trunc)} "
                  f"F={F} T={T} Y={Y:.2f} g={g:.3f} gbar={controller['gbar']:.3f} -> K_next={K_next} cooldown_next={cooldown_next}")

        # Store last attempt signals
        last_F, last_T, last_Y, last_g = F, T, Y, g
        last_items = items
        last_id_to_global = id_to_global
        last_packed_len = K_attempt

        # Warmup bookkeeping (only count successful batches)
        if (not controller["warmed"]) and (F == 0):
            controller["warmup_successes"] += 1

            # Per-item list tokens proxy: list-part tokens / K
            list_tokens = tok_len(json.dumps(packed, ensure_ascii=False))
            controller["warmup_per_item_list_tokens"].append(list_tokens / max(1, K_attempt))

            # When warmup completes, initialize K using rho0 (no latency-derived L_max)
            if controller["warmup_successes"] >= WARMUP_B0:
                per_item_med = float(sorted(controller["warmup_per_item_list_tokens"])[len(controller["warmup_per_item_list_tokens"]) // 2])

                Cin = C_in_budget()
                max_window = min(len(idxs), Ktok)
                packed_max, _ = pack_items(messages, idxs[:max_window])
                local_ctx_max = build_local_batch_context(packed_max, streamer)
                H = H_static_tokens(streamer_general_context, DEVELOPER_PROMPT, local_ctx_max)

                slack = max(0, Cin - H)
                K_init = int(math.floor((RHO0 * slack) / max(1e-9, per_item_med)))
                K_init = max(1, min(K_init, Ktok, MAX_K))

                controller["K"] = K_init
                controller["cooldown"] = 0
                controller["gbar"] = 0.0
                controller["warmed"] = True

                if PRINT_EVERY_BATCH:
                    print(f"[{now_ts()}] WARMUP DONE: per_item_med={per_item_med:.1f} | Cin={Cin} H={H} slack={slack} | K_init={K_init}")

        # If usable: apply results and advance pointer
        if F == 0:
            updated = apply_batch_results(messages, items, id_to_global)
            newly_annotated += updated
            success = True

            if SAVE_EVERY_BATCH:
                safe_write_json(WORK_MESSAGES_PATH, messages)

            # After success, commit controller state for next pointer using AIMD outputs
            controller["K"] = K_next
            controller["cooldown"] = cooldown_next
            break

        # If failure: do not advance pointer; set controller state for retry
        controller["K"] = K_next
        controller["cooldown"] = cooldown_next
        K_target = K_next  # retry uses reduced K

    # Batch-level logging + stats persistence
    batch_counter += 1
    stats["batches_completed"] = int(stats.get("batches_completed", 0)) + 1

    batch_log = {
        "batch_no": stats["batches_completed"],
        "ts": now_ts(),
        "streamer": streamer,
        "success": bool(success),
        "attempts_used": attempt,
        "K_token": int(Ktok),
        "K_last_attempt": int(last_packed_len),
        "last_F": int(last_F),
        "last_T": int(last_T),
        "last_Y": float(last_Y),
        "last_g": float(last_g),
        "controller_K_next": int(controller["K"]),
        "controller_cooldown_next": int(controller["cooldown"]),
        "last_latency_sec": float(last_dt),
        "last_output_tokens": int(last_out_tok),
    }
    stats.setdefault("batches", []).append(batch_log)

    # Print batch summary + cumulative tokens/cost/time
    if PRINT_EVERY_BATCH:
        if success:
            print(f"[{now_ts()}]   BATCH DONE: SUCCESS | updated≈{newly_annotated} (cumulative newly_annotated this run)")
        else:
            print(f"[{now_ts()}]   BATCH DONE: FAILURE | last_K={last_packed_len} (will force K=1 next)")

    # Persist stats (and cumulative elapsed) each batch
    # NOTE: save_stats adds (time.time()-session_start_wall) into total_elapsed_seconds; we then reset session_start_wall.
    save_stats(stats, session_start_wall)
    session_start_wall = time.time()  # reset session timer after persisting cumulative elapsed

    if PRINT_EVERY_BATCH:
        # after save_stats, cumulative elapsed already updated; show using a fresh session_start_wall (0 elapsed since reset)
        print_cum_stats_line(stats, session_start_wall, prefix="CUM_AFTER_BATCH")

    # Terminal failure handling (still failing at K=1)
    if not success:
        print(f"[{now_ts()}]   TERMINAL FAILURE at streamer='{streamer}' pointer. Forcing K=1 next (no pointer advance).")
        controller["K"] = 1
        controller["cooldown"] = N_COOLDOWN
        continue

# -----------------------------
# 11) Export XLSX + totals
# -----------------------------
df = pd.DataFrame(messages)
front = ["streamer", "msgTag", "raw_msg", "Annotated"] + BINARY_FIELDS + ["Dimension"]
cols = front + [c for c in df.columns if c not in front]
df = df[cols]
df.to_excel(XLSX_OUT, index=False)

remaining = sum(1 for m in messages if not is_annotated(m))

# Load latest persisted totals for reporting (already cumulative across resumes)
in_tok  = int(stats.get("total_input_tokens", 0))
out_tok = int(stats.get("total_output_tokens", 0))
c_tok   = int(stats.get("total_cached_tokens", 0))
unc_in  = max(0, in_tok - c_tok)
cost_usd = float(stats.get("total_cost_usd", 0.0))
elapsed_total = float(stats.get("total_elapsed_seconds", 0.0))

print(f"\n[{now_ts()}] DONE.")
print(f"[{now_ts()}] Total messages={total} | Newly annotated this run={newly_annotated} | Remaining={remaining}")
print(f"[{now_ts()}] Updated JSON: {WORK_MESSAGES_PATH}")
print(f"[{now_ts()}] Stats JSON:   {STATS_PATH}")
print(f"[{now_ts()}] XLSX:        {XLSX_OUT}")
print(f"[{now_ts()}] Total time (cumulative across resumes): {elapsed_total:.2f} seconds")
print(f"[{now_ts()}] Tokens: input={in_tok} (uncached≈{unc_in}, cached={c_tok}) | output={out_tok} | requests={int(stats.get('total_requests', 0))}")
print(f"[{now_ts()}] Total estimated spend (cumulative): ${cost_usd:.4f} USD")
